# Evaluation of trained models

## 1. Load base models & preparation

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Define model names and embeddings
models_and_embeddings = [
    ("stv1", SentenceTransformer("distiluse-base-multilingual-cased-v1"), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_sentence_placeholder.npy")),
    ("stv2", SentenceTransformer("distiluse-base-multilingual-cased-v2"), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v2_sentence_placeholder.npy")),
    ("mini", SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2"), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_miniL12v2_sentence_placeholder.npy")),
    ("mpnet", SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2"), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_mpnet_v2_sentence_placeholder.npy")),
    ("robbert", SentenceTransformer("NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers"), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_robbert2022_sentence_placeholder.npy")),
    ("e5", SentenceTransformer("intfloat/multilingual-e5-large-instruct", trust_remote_code=True), np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_e5.npy"))
]

from bertopic import BERTopic

base_models = {}
for name, embedding_model, embeddings in models_and_embeddings:
    path = f"/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/{name}_base"
    loaded_model = BERTopic.load(path, embedding_model=embedding_model)
    base_models[name] = loaded_model

print("Finished loading base models.")


/root/anaconda3/envs/my_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Finished loading base models.


In [2]:
import pickle

tokens_path = "/workspace/persistent/mijnidbcoachnlp/data/tokens/tokens_sentences.pkl"
dict_path = "/workspace/persistent/mijnidbcoachnlp/data/tokens/dictionary_sentences.pkl"
with open(tokens_path, "rb") as f:
    tokenized_texts = pickle.load(f)
with open(dict_path, "rb") as f:
    dictionary = pickle.load(f)

print(tokenized_texts[0])
print(dictionary.dfs)

['geachte', 'ibd', 'groep', 'is', 'mijn', 'uitslag', 'al', 'binnen', 'van', 'de', 'botscan', 'van', 'afgelopen', 'donderdag']
{6: 177, 8: 362, 7: 18, 9: 7973, 10: 5978, 11: 1153, 1: 2259, 2: 606, 12: 6839, 4: 16232, 3: 10, 0: 770, 5: 364, 29: 675, 31: 1679, 17: 1049, 32: 849, 22: 1036, 26: 2520, 25: 33, 24: 3020, 16: 66, 30: 2335, 15: 473, 20: 5042, 14: 33, 28: 4857, 21: 703, 18: 9476, 27: 129, 33: 2720, 13: 1, 23: 2712, 19: 206, 34: 2399, 39: 2940, 36: 23313, 37: 12, 35: 176, 38: 22, 40: 2326, 47: 582, 46: 650, 44: 1554, 41: 14, 45: 3834, 42: 5, 43: 1, 50: 70, 49: 3874, 48: 2055, 51: 832, 63: 584, 73: 259, 77: 4242, 52: 255, 72: 432, 79: 239, 69: 10511, 61: 169, 67: 143, 75: 55, 80: 237, 55: 3025, 62: 1061, 81: 103, 64: 326, 87: 374, 58: 528, 65: 2294, 83: 87, 88: 199, 56: 388, 84: 3, 74: 947, 53: 257, 85: 1298, 86: 2465, 59: 120, 89: 252, 70: 3508, 68: 8827, 78: 2866, 66: 301, 57: 6797, 60: 3169, 76: 326, 82: 54, 54: 9, 71: 1625, 90: 239, 104: 2157, 114: 7080, 96: 3, 99: 1736, 120: 4

In [5]:
# functions for evaluation
from gensim.models.coherencemodel import CoherenceModel
from octis.evaluation_metrics.diversity_metrics import TopicDiversity

def get_top_words(topic_model, top_n):
    """Extract top words for each topic from BERTopic (excluding outliers)."""
    topics = topic_model.get_topics()  # topics is a dict: {topic_num: [(word, score), ...]}

    top_words = []
    for topic_num, word_score_list in topics.items():
        if topic_num == -1:
            continue  # Skip outlier topic (-1)
        words = [word for word, _ in word_score_list[:top_n]]  # Get only the top N words
        top_words.append(words)

    return top_words


# Evaluate u_mass coherence
def get_c_v(topic_model, tokenized_texts, top_words, dictionary):
    """Calculate C_v coherence score."""
    coherence_model = CoherenceModel(
        topics=top_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()


# evaluate nmpi
def get_npmi(topic_model, tokenized_texts, top_words, dictionary):
    """Calculate NPMI coherence using Gensim."""

    coherence_model = CoherenceModel(
        topics=top_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_npmi'
    )
    return coherence_model.get_coherence()


def get_outlier_proportion(topic_model):
    """Calculate the percentage of documents labeled as outliers (topic=-1)."""
    topics = topic_model.topics_
    outlier_count = list(topics).count(-1)
    return outlier_count / len(topics)


def get_topic_diversity(top_words, topk=10):
    """
    Compute topic diversity using OCTIS's TopicDiversity metric.
    
    Args:
        top_words: List of lists, where each sublist contains top words for a topic 
                (e.g., [["word1", "word2"], ["word3", "word4"], ...]).
        topk: Number of top words to consider (must match the length of sublists in top_words).
    
    Returns:
        diversity_score: Float between 0 (low diversity) and 1 (high diversity).
    """
    metric = TopicDiversity(topk=topk)
    diversity_score = metric.score({"topics": top_words})  # OCTIS expects {"topics": ...}
    return diversity_score

def evaluate_model(topic_model, tokens, dictionary):
    results = {}
    top_words = get_top_words(topic_model, 10)
    # Coherence Scores
    if tokens is not None and dictionary is not None:
        npmi_score = get_npmi(topic_model, tokens, top_words, dictionary)
        c_v_score = get_c_v(topic_model, tokens, top_words, dictionary)
        results["c_v_coherence"] = c_v_score
        results['npmi_coherence'] = npmi_score
    else:
        results['c_v_coherence'] = None
        results['npmi_coherence'] = None
    
    # Outlier Proportion
    outlier_prop = get_outlier_proportion(topic_model)
    results['outlier_proportion'] = outlier_prop

    # Topic Diversity
    topic_diversity = get_topic_diversity(top_words)
    results['topic_diversity'] = topic_diversity

    # Top words
    
    print("\n--- Model Evaluation Metrics ---")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, (int, float)) and value is not None else f"{metric}: {value}")
    for topics in top_words[:4]:
        print(topics)
    print("--------------------------------\n")

    return results


## 2. Evaluate all base models

In [22]:
import os
from bertopic import BERTopic

all_results = {}

for key in base_models:        
    print(f"Evaluating Model: {key}")    
    model = base_models[key]
    results = evaluate_model(model, tokenized_texts, dictionary, embeddings)
    # Store results with a unique key
    all_results[key] = results

Evaluating Model: mpnet_base

--- Model Evaluation Metrics ---
c_v_coherence: 0.3598
npmi_coherence: -0.1399
outlier_proportion: 0.3721
topic_diversity: 0.5940
['huisarts', 'arts', 'dokter', 'mdl', 'ziekenhuis', 'verpleegkundige', 'medische', 'contact', 'behandeling', 'afspraak']
['medicatie', 'medicijn', 'medicijnen', 'bijwerkingen', 'bijwerking', 'medicijngebruik', 'zetpillen', 'dosering', 'stoppen', 'alternatief']
['buikpijn', 'buik', 'darmen', 'maag', 'last', 'buikkrampen', 'darm', 'opgezette', 'gevoel', 'pijn']
['calprotectine', 'calpro', 'thuistest', 'waarde', 'test', 'meten', 'gedaan', 'bekend', 'gemeten', 'bepaling']
--------------------------------

Evaluating Model: mini_base

--- Model Evaluation Metrics ---
c_v_coherence: 0.3627
npmi_coherence: -0.1359
outlier_proportion: 0.3425
topic_diversity: 0.6057
['huisarts', 'dokter', 'arts', 'mdl', 'ziekenhuis', 'specialist', 'behandelend', 'contact', 'doktersverklaring', 'medische']
['medicatie', 'zetpillen', 'medicijnen', 'medicij

/root/anaconda3/envs/my_env/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/root/anaconda3/envs/my_env/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



--- Model Evaluation Metrics ---
c_v_coherence: 0.3598
npmi_coherence: nan
outlier_proportion: 0.4737
topic_diversity: 0.5909
['poli', 'afspraak', 'telefonische', 'staan', 'consult', 'telefonisch', 'gepland', 'poliafspraak', 'belafspraak', 'aanstaande']
['slijm', 'bloed', 'ontlasting', 'soms', 'helder', 'bloedverlies', 'zit', 'erbij', 'afvegen', 'verlies']
['calprotectine', 'calpro', 'thuistest', 'waarde', 'test', 'gedaan', 'uitslag', 'bepaling', 'bekend', 'gemeten']
['endoscopie', 'darmonderzoek', 'coloscopie', 'colonscopie', 'scopie', 'gepland', 'colonoscopie', 'gastroscopie', 'afdeling', 'geopereerd']
--------------------------------



In [24]:
import pandas as pd 

all_results = pd.DataFrame(all_results)
all_results_t = all_results.T
all_results_t

,c_v_coherence,npmi_coherence,outlier_proportion,topic_diversity
mpnet_base,0.359826,-0.139940,0.372139,0.594022
mini_base,0.362681,-0.135929,0.342542,0.605738
stv1_base,0.350337,-0.162780,0.433157,0.634862
stv2_base,0.351342,-0.152627,0.396824,0.603040
robbert_base,0.360867,-0.143620,0.282862,0.639155
e5_base,0.359784,NaN,0.473723,0.590936


## 3. Observe trends in metrics when tuning hyperparameters

In [9]:
tuning_results = {}
min_cluster_sizes = list(range(5, 31, 5))
n_neighbors = list(range(5, 31, 5))
n_components = list(range(5, 11, 1))

### 3.1. min_cluster_size

In [12]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# Shared settings
bertopic_settings = {
    "umap_model": UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42),
    "hdbscan_model": HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=False),
    "vectorizer_model": CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'),
    "calculate_probabilities": False,
    "verbose": True
}

for name, embedding_model, embeddings in models_and_embeddings[0:1]:
    # configure the base settings
    topic_model = BERTopic(**bertopic_settings)

    for size in min_cluster_sizes:
        topic_model.hdbscan_model = HDBSCAN(min_cluster_size=size, metric='euclidean', cluster_selection_method='eom', prediction_data=False)

        print(f"Tuning model: {name} at min_cluster_size {size}")

        topics, probs = topic_model.fit_transform(sentences, embeddings)
        evaluation_results = evaluate_model(topic_model, tokens, dictionary)
        tuning_results[(name, size)] = {
            "name": name,
            "min_cluster_size": size,
            "results": evaluation_results
        }

Tuning model: stv1 at min_cluster_size 5


NameError: name 'sentences' is not defined